In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc networkx numpy qiskit-ibm-catalog sympy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Optimization Solver: una Qiskit Function de Q-CTRL Fire Opal
> **Note:** Las Qiskit Functions son una función experimental disponible únicamente para usuarios del Plan Premium, el Plan Flex y el Plan On-Prem (a través de la API de IBM Quantum Platform) de IBM Quantum&reg;. Se encuentran en estado de versión preliminar y están sujetas a cambios.

## Descripción general
Con el Fire Opal Optimization Solver, puedes resolver problemas de optimización a escala utilitaria en hardware cuántico sin necesidad de experiencia en computación cuántica. Simplemente introduce la definición del problema de alto nivel y el Solver se encarga del resto. Todo el flujo de trabajo es consciente del ruido y aprovecha la [Gestión del rendimiento de Fire Opal](/guides/q-ctrl-performance-management) internamente. El Solver ofrece de manera consistente soluciones precisas a problemas difíciles de resolver de forma clásica, incluso a escala de dispositivo completo en los QPUs IBM&reg; más grandes.

El Solver es flexible y puede utilizarse para resolver problemas de optimización combinatoria definidos como funciones objetivo o grafos arbitrarios. Los problemas no tienen que ser mapeados a la topología del dispositivo. Tanto los problemas sin restricciones como los restringidos son solucionables, siempre que las restricciones puedan formularse como términos de penalización. Los ejemplos incluidos en esta guía demuestran cómo resolver un problema de optimización a escala utilitaria sin restricciones y uno con restricciones, usando diferentes tipos de entrada del Solver. El primer ejemplo involucra un problema Max-Cut definido en un grafo 3-regular de 156 nodos, mientras que el segundo aborda un problema de Cobertura Mínima de Vértices de 50 nodos definido por una función de costo.

Para obtener acceso al Optimization Solver, [contacta a Q-CTRL](https://form.typeform.com/to/uOAVDnGg?typeform-source=q-ctrl.com).
## Descripción de la función
El Solver optimiza y automatiza por completo todo el algoritmo, desde la supresión de errores a nivel de hardware hasta el mapeo eficiente del problema y la optimización clásica en bucle cerrado. En segundo plano, el pipeline del Solver reduce los errores en cada etapa, habilitando el rendimiento mejorado necesario para escalar de manera significativa. El flujo de trabajo subyacente está inspirado en el Quantum Approximate Optimization Algorithm (QAOA), que es un algoritmo híbrido cuántico-clásico. Para un resumen detallado del flujo de trabajo completo del Optimization Solver, consulta [el manuscrito publicado](https://arxiv.org/abs/2406.01743).

![Visualización del flujo de trabajo del Optimization Solver](../docs/images/guides/qctrl-optimization/solver_workflow.svg)

Para resolver un problema genérico con el Optimization Solver:
1. Define tu problema como una función objetivo, un grafo o una cadena de espín `SparsePauliOp`.
2. Conéctate a la función a través del Catálogo de Funciones de Qiskit.
3. Ejecuta el problema con el Solver y recupera los resultados.
## Entradas y salidas
### Entradas
| Nombre    | Tipo                    | Descripción                                                                                                                                                                                         | Obligatorio | Predeterminado | Ejemplo |
| ---------  |-------------------------|-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------| -------- | ---------- | ---------- |
| problem  | `str` o `SparsePauliOp` | Una de las representaciones listadas en "Formatos de problema aceptados"                                                                                                                                  | Sí | N/A   |`Poly(2.0*n[0]*n[1] + 2.0*n[0]*n[2] - 3.13520241298341*n[0] + 2.0*n[1]*n[2] - 3.14469748506794*n[1] - 3.18897660121566*n[2] + 6.0, n[0], n[1], n[2], domain='RR')` |
| problem_type  | `str`                   | Nombre de la clase del problema; solo se usa para definiciones de problemas de grafo y cadena de espín, que se limitan a `"maxcut"` o `"spin_chain"`; no se requiere para definiciones de problemas de función objetivo arbitraria | No      | `None`| `"maxcut"`      |
| backend_name  | `str`                   | Nombre del backend a usar                                                                                                                                                                          | No  | El backend menos ocupado al que tu instancia tiene acceso    | `"ibm_fez"`      |
| options  | `dict`                  | Opciones de entrada, que incluyen: (Opcional) `session_id`: `str`; el comportamiento predeterminado crea una nueva sesión                                                                                      | No | `None`    | `{"session_id": "cw2q70c79ws0008z6g4g"}`     |

**Formatos de problema aceptados:**
- Representación en expresión polinómica de una función objetivo. Idealmente creada en Python con un objeto SymPy Poly existente y formateada como cadena de texto usando [sympy.srepr](https://docs.sympy.org/latest/tutorials/intro-tutorial/printing.html#srepr).
- Representación en grafo de un tipo de problema específico. El grafo debe crearse usando la librería networkx en Python. Luego debe convertirse a cadena de texto usando la función de networkx `[nx.readwrite.json_graph.adjacency_data](http://nx.readwrite.json_graph.adjacency_data.)`.
- Representación en cadena de espín de un problema específico. La cadena de espín debe representarse como un objeto `SparsePauliOp`; consulta la [documentación](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.SparsePauliOp) para más detalles.

**Backends soportados:**
Ejecuta el siguiente código para ver la lista de backends actualmente soportados. Si tu dispositivo no aparece en la lista, [contacta a Q-CTRL](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com) para añadir soporte.

In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()

service.backends()

[<IBMBackend('ibm_fez')>,
 <IBMBackend('ibm_brisbane')>,
 <IBMBackend('ibm_pittsburgh')>,
 <IBMBackend('ibm_kingston')>,
 <IBMBackend('ibm_torino')>,
 <IBMBackend('ibm_marrakesh')>]

**Options:**
| Name   | Type   | Description  | Default |
|--------|----------|-----------------------|---------------------|
| session_id | `str`  | An existing Qiskit Runtime session ID  | `"cw4r3je6f0t010870y3g"` |
| job_tags | `list[str]` | A list of job tags | `[]`|

### Outputs
| Name    | Type | Description | Example |
| ---------  | ---------------- | -------------------------- | -------- |
| result  | `dict[str, Any]`              | Solution and metadata listed under "Result dictionary contents"         | `{'solution_bitstring_cost': 3.0, ‘final_bitstring_distribution’: {‘000001’: 100, ‘000011’: 2}, ‘iteration_count’: 3, 'solution_bitstring': ‘000001’,  'variables_to_bitstring_index_map': {n[1]': 5, 'n[2]': 4, 'n[3]': 3, 'n[4]': 2, 'n[5]': 1}, 'best_parameters': [0.19628831763697097, -1.047052334523102], 'warnings': []}`     |


**Result dictionary contents:**
| Name    | Type | Description |
| ---------  | ---------------- | -------------------------- |
| solution_bitstring_cost  | `float`              | The best minimum cost across all iterations        |
| final_bitstring_distribution  | `CountsDict`              | The bitstring counts dictionary associated with the minimum cost        |
| solution_bitstring | `str`              | Bitstring with the best cost in the final distribution         |
| iteration_count  | `int`              | The total number of QAOA iterations performed by the Solver        |
| variables_to_bitstring_index_map  | `float`              | The mapping from the variables to the equivalent bit in the bitstring       |
| best_parameters  | `list[float]`            | The optimized beta and gamma parameters across all iterations  |
| warnings  |`list[str]`              | The warnings produced while compiling or running QAOA (defaults to None)   |

## Benchmarks

[Published benchmarking results](https://arxiv.org/abs/2406.01743) show that the Solver successfully solves problems with over 120 qubits, even outperforming previously published results on quantum annealing and trapped-ion devices. The following benchmark metrics provide a rough indication of the accuracy and scaling of problem types based on a few examples. Actual metrics may differ based on various problem features, such as the number of terms in the objective function (density) and their locality, number of variables, and polynomial order.

The "Number of qubits" indicated is not a hard limitation but represents rough thresholds where you can expect extremely consistent solution accuracy. Larger problem sizes have been successfully solved, and testing beyond these limits is encouraged.

Arbitrary qubit connectivity is supported across all problem types.

| Problem type    | Number of qubits | Example | Accuracy | Total time (s) | Runtime usage (s) | Number of iterations
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |---- |
| Sparsely-connected quadratic problems  | 156 | 3-regular Max-Cut| 100%     | 1764     | 293          | 16 |
| Higher-order binary optimization | 156 | Ising spin-glass model | 100%      | 1461     | 272          | 16 |
| Densely-connected quadratic problems | 50 | Fully-connected Max-Cut| 100%      |  1758    | 268  | 12 |
| Constrained problem with penalty terms | 50 | Weighted Minimum Vertex Cover with 8% edge density | 100%      | 1074     | 215 | 10 |

## Get started

First, authenticate using your [IBM Quantum API key](http://quantum.cloud.ibm.com/). Then, select the Qiskit Function as follows. (This snippet assumes you've already [saved your account](/docs/guides/functions#install-qiskit-functions-catalog-client) to your local environment.)

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Access Function
solver = catalog.load("q-ctrl/optimization-solver")

**Opciones:**
| Nombre   | Tipo   | Descripción  | Predeterminado |
|--------|----------|-----------------------|---------------------|
| session_id | `str`  | ID de sesión existente de Qiskit Runtime  | `"cw4r3je6f0t010870y3g"` |
| job_tags | `list[str]` | Lista de etiquetas de trabajo | `[]`|

### Salidas
| Nombre    | Tipo | Descripción | Ejemplo |
| ---------  | ---------------- | -------------------------- | -------- |
| result  | `dict[str, Any]`              | Solución y metadatos listados en "Contenido del diccionario de resultados"         | `{'solution_bitstring_cost': 3.0, 'final_bitstring_distribution': {'000001': 100, '000011': 2}, 'iteration_count': 3, 'solution_bitstring': '000001',  'variables_to_bitstring_index_map': {n[1]': 5, 'n[2]': 4, 'n[3]': 3, 'n[4]': 2, 'n[5]': 1}, 'best_parameters': [0.19628831763697097, -1.047052334523102], 'warnings': []}`     |

**Contenido del diccionario de resultados:**
| Nombre    | Tipo | Descripción |
| ---------  | ---------------- | -------------------------- |
| solution_bitstring_cost  | `float`              | El mejor costo mínimo en todas las iteraciones        |
| final_bitstring_distribution  | `CountsDict`              | El diccionario de conteos de cadenas de bits asociado con el costo mínimo        |
| solution_bitstring | `str`              | Cadena de bits con el mejor costo en la distribución final         |
| iteration_count  | `int`              | El número total de iteraciones QAOA realizadas por el Solver        |
| variables_to_bitstring_index_map  | `float`              | El mapeo de las variables al bit equivalente en la cadena de bits       |
| best_parameters  | `list[float]`            | Los parámetros beta y gamma optimizados en todas las iteraciones  |
| warnings  |`list[str]`              | Las advertencias producidas al compilar o ejecutar QAOA (por defecto es None)   |
## Benchmarks
Los [resultados de benchmarking publicados](https://arxiv.org/abs/2406.01743) muestran que el Solver resuelve con éxito problemas con más de 120 qubits, superando incluso resultados previamente publicados en recocido cuántico y dispositivos de iones atrapados. Las siguientes métricas de benchmark ofrecen una indicación aproximada de la precisión y escalabilidad de los tipos de problemas basándose en algunos ejemplos. Las métricas reales pueden variar según diversas características del problema, como el número de términos en la función objetivo (densidad) y su localidad, el número de variables y el orden polinómico.

El "Número de qubits" indicado no es una limitación estricta, sino que representa umbrales aproximados donde puedes esperar una precisión de solución extremadamente consistente. Se han resuelto con éxito problemas de mayor tamaño, y se alienta a hacer pruebas más allá de estos límites.

La conectividad arbitraria de qubits está soportada en todos los tipos de problemas.

| Tipo de problema    | Número de qubits | Ejemplo | Precisión | Tiempo total (s) | Uso de runtime (s) | Número de iteraciones
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |---- |
| Problemas cuadráticos con conectividad dispersa  | 156 | Max-Cut 3-regular| 100%     | 1764     | 293          | 16 |
| Optimización binaria de orden superior | 156 | Modelo de espín-vidrio de Ising | 100%      | 1461     | 272          | 16 |
| Problemas cuadráticos con conectividad densa | 50 | Max-Cut completamente conectado| 100%      |  1758    | 268  | 12 |
| Problema restringido con términos de penalización | 50 | Cobertura Mínima de Vértices ponderada con 8% de densidad de aristas | 100%      | 1074     | 215 | 10 |
## Primeros pasos
Primero, autentícate usando tu [clave API de IBM Quantum](http://quantum.cloud.ibm.com/). Luego, selecciona la Qiskit Function de la siguiente manera. (Este fragmento de código asume que ya has [guardado tu cuenta](/guides/functions#install-qiskit-functions-catalog-client) en tu entorno local.)

In [2]:
# %pip install networkx numpy

## Ejemplo: Optimización sin restricciones
Ejecuta el problema de [corte máximo](https://en.wikipedia.org/wiki/Maximum_cut) (Max-Cut). El siguiente ejemplo demuestra las capacidades del Solver en un problema Max-Cut de grafo 3-regular no ponderado de 156 nodos, aunque también puedes resolver problemas de grafos ponderados.
Además de `qiskit-ibm-catalog`, también necesitarás los siguientes paquetes para ejecutar este ejemplo: `networkx` y `numpy`. Puedes instalarlos descomentando la siguiente celda si estás ejecutando este ejemplo en un notebook con el kernel de IPython.

In [2]:
import networkx as nx
import numpy as np

# Generate a random graph with 156 nodes
maxcut_graph = nx.random_regular_graph(d=3, n=156, seed=8)

In [3]:
# Optionally, visualize the graph
nx.draw_networkx(
    maxcut_graph, nx.kamada_kawai_layout(maxcut_graph), node_size=100
)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/0a7255e1-0.avif" alt="Output of the previous code cell" />

The Solver accepts a string as the problem definition input.

In [4]:
# Convert graph to string
problem_as_str = nx.readwrite.json_graph.adjacency_data(maxcut_graph)

![Salida de la celda de código anterior](../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/0a7255e1-0.avif)

El Solver acepta una cadena de texto como entrada para la definición del problema.

In [9]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy(n_qubits=156).name

In [ ]:
# Solve the problem
maxcut_job = solver.run(
    problem=problem_as_str,
    problem_type="maxcut",
    backend_name=backend_name,  # E.g. "ibm_fez"
)

Check your Qiskit Function workload's [status](/docs/guides/functions#check-job-status) or return [results](/docs/guides/functions#retrieve-results) as follows:

In [14]:
# Get job status
print(maxcut_job.status())

QUEUED


Comprueba el [estado](/guides/functions#check-job-status) de la carga de trabajo de tu Qiskit Function o recupera los [resultados](/guides/functions#retrieve-results) de la siguiente manera:

In [15]:
# Poll for results
maxcut_result = maxcut_job.result()

# Take the absolute value of the solution since the cost function is minimized
qctrl_maxcut = abs(maxcut_result["solution_bitstring_cost"])

# Print the optimal cut value found by the Optimization Solver
print(f"Optimal cut value: {qctrl_maxcut}")

Optimal cut value: 209.0


You can verify the accuracy of the result by solving the problem classically with open-source solvers like [PuLP](https://coin-or.github.io/pulp/) if the graph is not densely connected. High density problems may require advanced classical solvers to validate the solution.

## Example: Constrained optimization
The prior Max-Cut example is a common quadratic unconstrained binary optimization problem. Q-CTRL's Optimization Solver can be used for various problem types, including constrained optimization. You can solve arbitrary problem types by inputting the problem definition represented as a polynomial where constraints are modeled as penalty terms.

The following example demonstrates how to construct a cost function for a constrained optimization problem, [minimum vertex cover](https://en.wikipedia.org/wiki/Vertex_cover) (MVC).

In addition to the `qiskit-ibm-catalog` and `qiskit` packages, you will also use the following packages to run this example: `numpy`, `networkx`, and `sympy`. You can install these packages by uncommenting the following cell if you are running this example in a notebook using the IPython kernel.

In [ ]:
# %pip install numpy networkx sympy

### 3. Recupera el resultado
Recupera el valor de corte óptimo del diccionario de resultados.

> **Note:** El mapeo de las variables a la cadena de bits puede haber cambiado. El diccionario de salida contiene un subdicionario `variables_to_bitstring_index_map` que ayuda a verificar el orden.

In [ ]:
import networkx as nx
from sympy import symbols, Poly, srepr

# To change the weights, change the seed to any integer.
rng_seed = 18
_rng = np.random.default_rng(rng_seed)
node_count = 50
edge_probability = 0.08
mvc_graph = nx.erdos_renyi_graph(
    node_count, edge_probability, seed=rng_seed, directed=False
)

# add node weights
for i in mvc_graph.nodes:
    mvc_graph.add_node(i, weight=_rng.random())

# Optionally, visualize the graph
nx.draw_networkx(mvc_graph, nx.kamada_kawai_layout(mvc_graph), node_size=200)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.avif" alt="Output of the previous code cell" />

A standard optimization model for weighted MVC can be formulated as follows. First, a penalty must be added for any case where an edge is not connected to a vertex in the subset. Therefore, let $n_i = 1$ if vertex $i$ is in the cover (i.e., in the subset) and $n_i = 0$ otherwise. Second, the goal is to minimize the total number of vertices in the subset, which can be represented by the following function:

$$\textbf{Minimize}\qquad y = \sum_{i\in V} \omega_i n_i$$

In [ ]:
# Construct the cost function.
variables = symbols([f"n[{i}]" for i in range(node_count)])
cost_function = Poly(0, variables)

for i in mvc_graph.nodes():
    weight = mvc_graph.nodes[i].get("weight", 0)
    cost_function += variables[i] * weight

Puedes verificar la precisión del resultado resolviendo el problema de forma clásica con solvers de código abierto como [PuLP](https://coin-or.github.io/pulp/) si el grafo no tiene conectividad densa. Los problemas de alta densidad pueden requerir solvers clásicos avanzados para validar la solución.
## Ejemplo: Optimización con restricciones
El ejemplo anterior de Max-Cut es un problema de optimización binaria cuadrática sin restricciones común. El Optimization Solver de Q-CTRL puede usarse para distintos tipos de problemas, incluida la optimización con restricciones. Puedes resolver tipos de problemas arbitrarios introduciendo la definición del problema representada como un polinomio donde las restricciones se modelan como términos de penalización.

El siguiente ejemplo demuestra cómo construir una función de costo para un problema de optimización con restricciones: la [cobertura mínima de vértices](https://en.wikipedia.org/wiki/Vertex_cover) (MVC, por sus siglas en inglés).
Además de los paquetes `qiskit-ibm-catalog` y `qiskit`, también necesitarás los siguientes paquetes para ejecutar este ejemplo: `numpy`, `networkx` y `sympy`. Puedes instalarlos descomentando la siguiente celda si estás ejecutando este ejemplo en un notebook con el kernel de IPython.

In [ ]:
# Add penalty term.
penalty_constant = 2
for i, j in mvc_graph.edges():
    cost_function += penalty_constant * (
        1 - variables[i] - variables[j] + variables[i] * variables[j]
    )

### 1. Define el problema
Define un problema MVC aleatorio generando un grafo con nodos de peso aleatorio.

In [ ]:
# Solve the problem
mvc_job = solver.run(
    problem=srepr(cost_function),
    backend_name=backend_name,  # E.g. "ibm_fez"
)

![Salida de la celda de código anterior](../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.avif)

Un modelo de optimización estándar para MVC ponderado puede formularse de la siguiente manera. Primero, debe añadirse una penalización para cualquier caso en que una arista no esté conectada a un vértice del subconjunto. Por lo tanto, sea $n_i = 1$ si el vértice $i$ está en la cobertura (es decir, en el subconjunto) y $n_i = 0$ en caso contrario. Segundo, el objetivo es minimizar el número total de vértices en el subconjunto, lo cual puede representarse mediante la siguiente función:

$$\textbf{Minimize}\qquad y = \sum_{i\in V} \omega_i n_i$$

In [ ]:
print(mvc_job.status())

Ahora, cada arista del grafo debe incluir al menos un extremo de la cobertura, lo cual puede expresarse como la desigualdad:

$$n_i + n_j \ge 1 \texttt{ for all } (i,j)\in E$$

Cualquier caso en que una arista no esté conectada al vértice de la cobertura debe ser penalizado. Esto puede representarse en la función de costo añadiendo una penalización de la forma $P(1-n_i-n_j+n_i n_j)$ donde $P$ es una constante de penalización positiva. Por lo tanto, una alternativa sin restricciones a la desigualdad restringida para MVC ponderado es:

$$\textbf{Minimize}\qquad y = \sum_{i\in V}\omega_i n_i + P(\sum_{(i,j)\in E}(1 - n_i - n_j + n_i n_j))$$

In [ ]:
mvc_result = mvc_job.result()
qctrl_cost = mvc_result["solution_bitstring_cost"]

# Print results
print(f"Solution cost: {qctrl_cost}")

Solution cost: 10.248198273708624


### 2. Ejecuta el problema